In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters pypdf faiss-cpu sentence-transformers transformers

print(" All dependencies installed successfully!")
print("Packages: langchain, faiss-cpu, pypdf, sentence-transformers, transformers")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
 All dependencies installed successfully!
Packages: langchain, faiss-cpu, pypdf, sentence-transformers, transformers


In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
import torch

print(" All libraries imported successfully!")

 All libraries imported successfully!


In [ ]:
print(" Please upload your PDF document (one file recommended)")
uploaded = files.upload()

if uploaded:
    pdf_filename = list(uploaded.keys())[0]
    print(f" Uploaded: {pdf_filename}")
else:
    raise ValueError("No file uploaded. Please run this cell again and upload a PDF.")

 Please upload your PDF document (one file recommended)


Saving RAG_Research_Paper.pdf to RAG_Research_Paper.pdf
 Uploaded: RAG_Research_Paper.pdf


In [ ]:
loader = PyPDFLoader(pdf_filename)
docs = loader.load()

print(f" PDF loaded: {len(docs)} pages")
if docs:
    print(f"Preview (first 300 chars): {docs[0].page_content[:300]}...")

 PDF loaded: 21 pages
Preview (first 300 chars): 1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng
Wangc, and Haofen Wang a,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai K...


In [ ]:
# Clean each document in-place
for doc in docs:
    # Remove extra whitespace and normalize
    cleaned_content = " ".join(doc.page_content.split())
    doc.page_content = cleaned_content.strip()

print(" Text cleaning completed")
print(f"Total cleaned characters: {sum(len(doc.page_content) for doc in docs):,}")
print(f"Preview (first document): {docs[0].page_content[:400]}...")

✅ Text cleaning completed
Total cleaned characters: 109,932
Preview (first document): 1 Retrieval-Augmented Generation for Large Language Models: A Survey Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng Wangc, and Haofen Wang a,c aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University cCollege of Design and I...


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(docs)

print(f" Chunking completed: {len(chunks)} chunks created")
print(f"First chunk preview: {chunks[0].page_content[:300]}...")

 Chunking completed: 139 chunks created
First chunk preview: 1 Retrieval-Augmented Generation for Large Language Models: A Survey Yunfan Gaoa, Yun Xiong b, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bi c, Yi Dai a, Jiawei Sun a, Meng Wangc, and Haofen Wang a,c aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University bShanghai K...


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(" Embedding model loaded (sentence-transformers/all-MiniLM-L6-v2)")
# Quick validation
test_embedding = embeddings.embed_query("This is a test sentence.")
print(f"Embedding dimension: {len(test_embedding)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding model loaded (sentence-transformers/all-MiniLM-L6-v2)
Embedding dimension: 384


In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings)

print(f" FAISS vector store created with {len(chunks)} chunks")
print("Index ready for similarity search")

 FAISS vector store created with 139 chunks
Index ready for similarity search


In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

def get_relevant_chunks(query: str):
    """Returns top-k relevant documents for a query."""
    return retriever.invoke(query)

print(" Retriever ready (top-3 similarity search)")
print("Retrieval function 'get_relevant_chunks' is available")

 Retriever ready (top-3 similarity search)
Retrieval function 'get_relevant_chunks' is available


In [ ]:
# Initialize local LLM (runs on CPU/GPU in Colab, no API key)
pipe = pipeline(
    task="text-generation",
    model="google/flan-t5-small",
    max_new_tokens=256,
    temperature=0.0,
    do_sample=False,
    device=0 if torch.cuda.is_available() else -1
)

llm = HuggingFacePipeline(pipeline=pipe)

print("Local LLM initialized (google/flan-t5-small)")
print("RAG pipeline is now ready to combine retrieval + generation")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cp

Local LLM initialized (google/flan-t5-small)
RAG pipeline is now ready to combine retrieval + generation


/tmp/ipykernel_9798/2200260047.py:11: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful, accurate, and truthful AI assistant focused on Responsible AI.
Use ONLY the provided context to answer the question.
If the answer is not in the context, say "I don't have enough information to answer this."
Do not hallucinate or make up information.

Context:
{context}

Question: {question}

Answer:"""
)

print("Context-aware prompt template created")
print("Ready for use in the final query cell")

Context-aware prompt template created
Ready for use in the final query cell


In [22]:
# User query (you can change this or use the input box)
query = input("Enter your question about the PDF: ") or "Summarize the main topic of this document"

print(f"\n Processing query: {query}")

# Step 1: Retrieve relevant chunks
retrieved_docs = get_relevant_chunks(query)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# Step 2: Print retrieved chunks for debugging / transparency (xAI / rAI best practice)
print("\n" + "="*60)
print(" RETRIEVED CHUNKS (for debugging and explainability)")
print("="*60)
for i, doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1} (source: {doc.metadata.get('source', 'N/A')}, page: {doc.metadata.get('page', 'N/A')}):")
    print(doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content)
    print("-" * 40)

# Step 3: Format prompt with context
formatted_prompt = prompt_template.format(context=context, question=query)

# Step 4: Generate final response using RAG
response = llm.invoke(formatted_prompt)

print("\n" + "="*60)
print(" FINAL RAG RESPONSE")
print("="*60)
print(response.strip())

print("\n RAG pipeline completed successfully!")
print("   • Hallucinations reduced by retrieved context")
print("   • Full transparency with printed chunks")
print("   • Ready for next query — just re-run this cell!")

Enter your question about the PDF: RAG

 Processing query: RAG

 RETRIEVED CHUNKS (for debugging and explainability)
Chunk 1 (source: RAG_Research_Paper.pdf, page: 0):
. This survey endeavors to fill this gap by mapping out the RAG process and charting its evolution and anticipated future paths, with a focus on the integration of RAG within LLMs. This paper considers both technical paradigms and research methods, summarizing three main research paradigms from over 100 RAG studies, and analyzing key technologies in the core stages of “Retrieval,” “Generation,” an...
----------------------------------------
Chunk 2 (source: RAG_Research_Paper.pdf, page: 1):
. The RAG research paradigm is continuously evolving, and we categorize it into three stages: Naive RAG, Advanced RAG, and Modular RAG, as showed in Figure 3. Despite RAG method are cost-effective and surpass the performance of the native LLM, they also exhibit several limitations. The development of Advanced RAG and Modular RAG is a 

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 FINAL RAG RESPONSE
You are an expert AI assistant specialized in accurate, context-grounded responses.

STRICT RULES:
- Answer ONLY using the information provided in the context below.
- If the answer is not present in the context, respond with: "I don't have enough information in the provided context to answer this question."
- Do NOT hallucinate, invent facts, or use external knowledge.
- Be concise, factual, and direct.

Context:
. This survey endeavors to fill this gap by mapping out the RAG process and charting its evolution and anticipated future paths, with a focus on the integration of RAG within LLMs. This paper considers both technical paradigms and research methods, summarizing three main research paradigms from over 100 RAG studies, and analyzing key technologies in the core stages of “Retrieval,” “Generation,” and “Augmentation.” On the other hand, current research tends to focus more on methods, lacking analysis and summarization of how to evaluate RAG. This paper compr